# Report Validations

Live demo notebook for Report Maintenance focused on Best Practice Analyzer (BPA) and Identifying Broken Reports

In [ ]:
%pip install -U semantic-link-labs

In [ ]:
import sempy_labs.report as rep
import pandas as pd

## What Is Report BPA?

Report BPA checks a Power BI report against a rules table to detect design, performance, and maintainability issues.

Each violation includes rule metadata and object context (for example page, visual, filter, or semantic model reference) so fixes can be prioritized quickly.

In [ ]:
# Update for your demo
report_workspace = "Demo"
report_name = "card_udf_pbir"

In [ ]:
# Run Report BPA with built-in standard rules
report_bpa_default_df = rep.run_report_bpa(
    report=report_name,
    workspace=report_workspace,
    return_dataframe=True
)

display(report_bpa_default_df.head(20))
print(f"Total report violations: {len(report_bpa_default_df)}")

## Custom Rule Format

Custom rules are passed as a pandas DataFrame with these columns:

- Category
- Scope
- Severity
- Rule Name
- Expression
- Description
- URL

| Column | Allowed Options | Notes |
|---|---|---|
| Severity | `Error`, `Warning`, `Info` | Fixed set |
| Scope | `Custom Visual`, `Page`, `Visual`, `Report Filter`, `Page Filter`, `Visual Filter`, `Report Level Measure`, `Semantic Model` | Can be a single value or a list |
| Category | Any text | Free-form |
| Rule Name | Any text | Free-form |
| Description | Any text | Free-form |
| URL | Any text/URL | Free-form |
| Expression | Python lambda | Returns a boolean mask used to select violating rows |

### How the Expression Works

The Expression column contains the rule logic used by Report BPA to evaluate each report scope DataFrame.

Each expression is usually a Python lambda with the signature `lambda df: <condition>`.

- df is the DataFrame for the selected Scope (for example Visual, Page Filter, or Semantic Model).
- The expression should return a boolean Series (row-level mask).

The expression must identify violating rows:
- `True`: the row violates the rule and appears in BPA results.
- `False`: the row passes the rule.

Think of it as a filter for anti-patterns in each report scope: if the condition detects a problem row, return `True`.

Examples:
- `lambda df: df["Valid Semantic Model Object"] == False`
- `lambda df: df["Visual Object Count"] > 5`
- `lambda df: df["Custom Visual Name"].notna() & (df["Custom Visual Name"] != "")`

Keep expressions deterministic and lightweight so scans remain fast and predictable.


In [ ]:
rpt = rep.ReportWrapper(report=report_name, workspace=report_workspace)
actual_base_theme_name = (
    rpt.get(file_path="definition/report.json")
    .get("themeCollection", {})
    .get("baseTheme", {})
    .get("name", "")
    .removesuffix(".json")
)
print(actual_base_theme_name)

required_base_theme_name = "CY25SU110"
print(required_base_theme_name)

In [ ]:
import pandas as pd

custom_report_rules = pd.DataFrame(
    [
        (
            "Governance",
            "Page",
            "Error",
            "Require a page named 'Help'",
            lambda df: (df.index == df.index.min())
            & (~df["Page Display Name"].fillna("").str.strip().str.casefold().eq("help").any()),
            "Each report must include a Help page.",
            "",
        ),
        (
            "Governance",
            "Page",
            "Error",
            f"Require base theme '{required_base_theme_name}'",
            lambda df: df.index.isin(df.head(1).index)
            & (actual_base_theme_name.strip().casefold() != required_base_theme_name.strip().casefold()),
            "Each report must use the required base theme.",
            "",
        )
    ],
    columns=[
        "Category",
        "Scope",
        "Severity",
        "Rule Name",
        "Expression",
        "Description",
        "URL",
    ],
)

report_bpa_custom_df = rep.run_report_bpa(
    report=report_name,
    workspace=report_workspace,
    rules=custom_report_rules,
    return_dataframe=True,
    export=False,
 )

display(report_bpa_custom_df)

## Detecting Broken Reports

In [ ]:
report = "card_udf_pbir_broken"
workspace = "Demo"

rpt = rep.ReportWrapper(report=report, workspace=workspace)
df = rpt.list_semantic_model_objects(extended=True)
df[df['Valid Semantic Model Object'] == False]



In [ ]:
report = "card_udf_broken"
workspace = "Demo"

rpt = rep.ReportWrapper(report=report, workspace=workspace)
df = rpt.list_semantic_model_objects(extended=True)
df[df['Valid Semantic Model Object'] == False]



In [ ]:
import json
import pandas as pd
import sempy_labs as labs
from sempy_labs.report import get_report_json, launch_report
from IPython.display import display

report_name       = "card_udf"   
report_workspace  = "Demo" 
dataset_name      = "card_udf"  
dataset_workspace = "Demo"  

# Render the report inline so you can see the current visual state.
launch_report(report=report_name, workspace=report_workspace)

# Build the valid (table, field) set from the target semantic model.
model_objects = labs.list_semantic_model_objects(dataset=dataset_name, workspace=dataset_workspace)
valid_set = set(zip(model_objects["Parent Name"], model_objects["Object Name"]))

# Retrieve the report layout JSON and extract every field reference it contains.
report_json = get_report_json(report=report_name, workspace=report_workspace)


def _find_entity_in_expression(obj):
    """Walk an Expression subtree to locate the SourceRef Entity name."""
    if not isinstance(obj, dict):
        return None
    if "SourceRef" in obj and isinstance(obj["SourceRef"], dict):
        return obj["SourceRef"].get("Entity")
    for v in obj.values():
        result = _find_entity_in_expression(v)
        if result:
            return result
    return None


def _extract_refs(obj, refs=None):
    """
    Recursively collect (table, field) pairs from the report JSON.

    report.json stores page/visual content as double-serialised JSON strings,
    so string values are parsed with json.loads before recursing.
    """
    if refs is None:
        refs = set()

    if isinstance(obj, dict):
        # Pattern 1: {"Entity": "...", "Property": "..."}
        if "Entity" in obj and "Property" in obj:
            refs.add((obj["Entity"], obj["Property"]))
        # Pattern 2: {"Property": "...", "Expression": {... SourceRef ...}}
        elif "Property" in obj and "Expression" in obj:
            entity = _find_entity_in_expression(obj["Expression"])
            if entity:
                refs.add((entity, obj["Property"]))
        # Pattern 3: {"queryRef": "Table.Field"}
        if "queryRef" in obj and isinstance(obj["queryRef"], str) and "." in obj["queryRef"]:
            table, _, field = obj["queryRef"].partition(".")
            refs.add((table, field))
        for v in obj.values():
            _extract_refs(v, refs)

    elif isinstance(obj, list):
        for item in obj:
            _extract_refs(item, refs)

    elif isinstance(obj, str):
        # Double-serialised blobs — parse and recurse.
        if len(obj) > 2 and obj[0] in ('{', '['):
            try:
                _extract_refs(json.loads(obj), refs)
            except (json.JSONDecodeError, ValueError):
                pass

    return refs


all_refs = _extract_refs(report_json)
broken_rows = [
    {"Table Name": entity, "Object Name": prop}
    for entity, prop in sorted(all_refs)
    if (entity, prop) not in valid_set
]
broken_objects = pd.DataFrame(broken_rows)

if broken_objects.empty:
    print("Validation passed — no broken references found.")
else:
    print(f"Validation failed — {len(broken_objects)} broken reference(s) found.")
    display(broken_objects)
    raise ValueError("Broken report detected: one or more semantic model references no longer resolve.")


In [ ]:
import json
import pandas as pd
import sempy_labs as labs
from sempy_labs.report import get_report_json, launch_report
from IPython.display import display

report_name       = "card_udf_broken"   
report_workspace  = "Demo" 
dataset_name      = "card_udf_broken"  
dataset_workspace = "Demo"  

# Render the report inline so you can see the current visual state.
launch_report(report=report_name, workspace=report_workspace)

# Build the valid (table, field) set from the target semantic model.
model_objects = labs.list_semantic_model_objects(dataset=dataset_name, workspace=dataset_workspace)
valid_set = set(zip(model_objects["Parent Name"], model_objects["Object Name"]))

# Retrieve the report layout JSON and extract every field reference it contains.
report_json = get_report_json(report=report_name, workspace=report_workspace)


def _find_entity_in_expression(obj):
    """Walk an Expression subtree to locate the SourceRef Entity name."""
    if not isinstance(obj, dict):
        return None
    if "SourceRef" in obj and isinstance(obj["SourceRef"], dict):
        return obj["SourceRef"].get("Entity")
    for v in obj.values():
        result = _find_entity_in_expression(v)
        if result:
            return result
    return None


def _extract_refs(obj, refs=None):
    """
    Recursively collect (table, field) pairs from the report JSON.

    report.json stores page/visual content as double-serialised JSON strings,
    so string values are parsed with json.loads before recursing.
    """
    if refs is None:
        refs = set()

    if isinstance(obj, dict):
        # Pattern 1: {"Entity": "...", "Property": "..."}
        if "Entity" in obj and "Property" in obj:
            refs.add((obj["Entity"], obj["Property"]))
        # Pattern 2: {"Property": "...", "Expression": {... SourceRef ...}}
        elif "Property" in obj and "Expression" in obj:
            entity = _find_entity_in_expression(obj["Expression"])
            if entity:
                refs.add((entity, obj["Property"]))
        # Pattern 3: {"queryRef": "Table.Field"}
        if "queryRef" in obj and isinstance(obj["queryRef"], str) and "." in obj["queryRef"]:
            table, _, field = obj["queryRef"].partition(".")
            refs.add((table, field))
        for v in obj.values():
            _extract_refs(v, refs)

    elif isinstance(obj, list):
        for item in obj:
            _extract_refs(item, refs)

    elif isinstance(obj, str):
        # Double-serialised blobs — parse and recurse.
        if len(obj) > 2 and obj[0] in ('{', '['):
            try:
                _extract_refs(json.loads(obj), refs)
            except (json.JSONDecodeError, ValueError):
                pass

    return refs


all_refs = _extract_refs(report_json)
broken_rows = [
    {"Table Name": entity, "Object Name": prop}
    for entity, prop in sorted(all_refs)
    if (entity, prop) not in valid_set
]
broken_objects = pd.DataFrame(broken_rows)

if broken_objects.empty:
    print("Validation passed — no broken references found.")
else:
    print(f"Validation failed — {len(broken_objects)} broken reference(s) found.")
    display(broken_objects)
    raise ValueError("Broken report detected: one or more semantic model references no longer resolve.")
